### Step 1: Download the Github data

In [36]:
import io
from typing import Iterable, Callable
import zipfile
import traceback
from dataclasses import dataclass

import requests


@dataclass
class RawRepositoryFile:
    filename: str
    content: str


class GithubRepositoryDataReader:
    """
    Downloads and parses markdown and code files from a GitHub repository.
    """

    def __init__(self,
                repo_owner: str,
                repo_name: str,
                allowed_extensions: Iterable[str] | None = None,
                filename_filter: Callable[[str], bool] | None = None
        ):
        """
        Initialize the GitHub repository data reader.
        
        Args:
            repo_owner: The owner/organization of the GitHub repository
            repo_name: The name of the GitHub repository
            allowed_extensions: Optional set of file extensions to include
                    (e.g., {"md", "py"}). If not provided, all file types are included
            filename_filter: Optional callable to filter files by their path
        """
        prefix = "https://codeload.github.com"
        self.url = (
            f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"
        )

        if allowed_extensions is not None:
            self.allowed_extensions = {ext.lower() for ext in allowed_extensions}

        if filename_filter is None:
            self.filename_filter = lambda filepath: True
        else:
            self.filename_filter = filename_filter

    def read(self) -> list[RawRepositoryFile]:
        """
        Download and extract files from the GitHub repository.
        
        Returns:
            List of RawRepositoryFile objects for each processed file
            
        Raises:
            Exception: If the repository download fails
        """
        resp = requests.get(self.url)
        if resp.status_code != 200:
            raise Exception(f"Failed to download repository: {resp.status_code}")

        zf = zipfile.ZipFile(io.BytesIO(resp.content))
        repository_data = self._extract_files(zf)
        zf.close()

        return repository_data

    def _extract_files(self, zf: zipfile.ZipFile) -> list[RawRepositoryFile]:
        """
        Extract and process files from the zip archive.
        
        Args:
            zf: ZipFile object containing the repository data

        Returns:
            List of RawRepositoryFile objects for each processed file
        """
        data = []

        for file_info in zf.infolist():
            filepath = self._normalize_filepath(file_info.filename)

            if self._should_skip_file(filepath):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode("utf-8", errors="ignore")
                    if content is not None:
                        content = content.strip()

                    file = RawRepositoryFile(
                        filename=filepath,
                        content=content
                    )
                    data.append(file)

            except Exception as e:
                print(f"Error processing {file_info.filename}: {e}")
                traceback.print_exc()
                continue

        return data

    def _should_skip_file(self, filepath: str) -> bool:
        """
        Determine whether a file should be skipped during processing.
        
        Args:
            filepath: The file path to check
            
        Returns:
            True if the file should be skipped, False otherwise
        """
        filepath = filepath.lower()

        # directory
        if filepath.endswith("/"):
            return True

        # hidden file
        filename = filepath.split("/")[-1]
        if filename.startswith("."):
            return True

        if self.allowed_extensions:
            ext = self._get_extension(filepath)
            if ext not in self.allowed_extensions:
                return True

        if not self.filename_filter(filepath):
            return True

        return False

    def _get_extension(self, filepath: str) -> str:
        """
        Extract the file extension from a filepath.
        
        Args:
            filepath: The file path to extract extension from
            
        Returns:
            The file extension (without dot) or empty string if no extension
        """
        filename = filepath.lower().split("/")[-1]
        if "." in filename:
            return filename.rsplit(".", maxsplit=1)[-1]
        else:
            return ""

    def _normalize_filepath(self, filepath: str) -> str:
        """
        Removes the top-level directory from the file path inside the zip archive.
        'repo-main/path/to/file.py' -> 'path/to/file.py'
        
        Args:
            filepath: The original filepath from the zip archive
            
        Returns:
            The normalized filepath with top-level directory removed
        """
        parts = filepath.split("/", maxsplit=1)
        if len(parts) > 1:
            return parts[1]
        else:
            return parts[0]

In [2]:
def read_github_data():
    repo_owner = 'DataTalksClub'
    repo_name = 'datatalksclub.github.io'
    
    allowed_extensions = {"md", "mdx"}

    reader = GithubRepositoryDataReader(
        repo_owner,
        repo_name,
        allowed_extensions=allowed_extensions
    )
    
    return reader.read()

In [3]:
github_data = read_github_data()

### Step 2: Filter only podcast files

In [4]:
podcast_files = [f for f in github_data if  f.filename.startswith("_podcast/") and "template" not in f.filename]

In [ ]:
### Number of podcast documents
len(podcast_files)

184

**Q: Number of podcast documents ?**  
**A:** There are 184 podcast files.

### Step 3: Chunking the data

In [7]:
def get_metadata(raw_data: dict, file: RawRepositoryFile) -> dict:
    
    metadata = {
        'filename': file.filename,
        'episode_title': raw_data.get('title'),
        'episode_number': raw_data.get('episode'),
        'ids': raw_data.get('ids'),
        'image': raw_data.get('image'),
        'links': raw_data.get('links'),
        'season': raw_data.get('season'),
        'short': raw_data.get('short'),
        'guests': raw_data.get('guests'),
    }
    
    return metadata

In [176]:
import frontmatter
for f in podcast_files:
    post = frontmatter.loads(f.content)
    raw_data = post.to_dict()

    data = [line.get('line') for line in raw_data['transcript']]
    metadata = get_metadata(raw_data, f)
    
    break  # just process the first file for demonstration

In [177]:
metadata

{'filename': '_podcast/_s12e08.md',
 'episode_title': 'The Journey of a Data Generalist: From Bioinformatics to Freelancing',
 'episode_number': 8,
 'ids': {'anchor': 'The-Journey-of-a-Data-Generalist-From-Bioinformatics-to-Freelancing---Jekaterina-Kokatjuhha-e1upvim',
  'youtube': 'FRi0SUtxdMw'},
 'image': 'images/podcast/s12e08-journey-of-data-generalist-from-bioinformatics-to-freelancing.jpg',
 'links': {'anchor': 'https://anchor.fm/datatalksclub/episodes/The-Journey-of-a-Data-Generalist-From-Bioinformatics-to-Freelancing---Jekaterina-Kokatjuhha-e1upvim',
  'apple': 'https://podcasts.apple.com/us/podcast/the-journey-of-a-data-generalist-from/id1541710331?i=1000599125044',
  'spotify': 'https://open.spotify.com/episode/5fB185hGlGYQmdk0kbIsPv?si=YtnsaYNzTc-fl7emZ2IjEA',
  'youtube': 'https://www.youtube.com/watch?v=FRi0SUtxdMw'},
 'season': 12,
 'short': 'The Journey of a Data Generalist: From Bioinformatics to Freelancing',
 'guests': ['jekaterinakokatjuhha']}

In [184]:
def parse_documents(files: list[RawRepositoryFile]) -> list[dict]:

    parsed_data = []

    for f in podcast_files:
        post = frontmatter.loads(f.content)
        raw_data = post.to_dict()

        if raw_data.get('transcript') is None:
            text = raw_data.get('content')
            metadata = get_metadata(raw_data, f)
        else:
            text = [line.get('line') for line in raw_data['transcript']]
            metadata = get_metadata(raw_data, f)
        
        chunk = {
            'text': text,
            'metadata': metadata}

        parsed_data.append(chunk)

    return parsed_data

In [185]:
parsed_data = parse_documents(podcast_files)

In [215]:
from typing import Any, Dict, Iterable, List


def sliding_window(
        seq: Iterable[Any],
        size: int,
        step: int
    ) -> List[Dict[str, Any]]:
    """
    Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size (int): The size of each chunk/window.
        step (int): The step size between consecutive windows.

    Returns:
        list: A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size]
        result.append({'start': i, 'content': batch})
        if i + size > n:
            break

    return result


def chunk_documents(
        documents: Iterable[Dict[str, str]],
        size: int = 2000,
        step: int = 1000,
        content_field_name: str = 'content'
) -> List[Dict[str, str]]:
    """
    Split a collection of documents into smaller chunks using sliding windows.

    Takes documents and breaks their content into overlapping chunks while preserving
    all other document metadata (filename, etc.) in each chunk.

    Args:
        documents: An iterable of document dictionaries. Each document must have a content field.
        size (int, optional): The maximum size of each chunk. Defaults to 2000.
        step (int, optional): The step size between chunks. Defaults to 1000.
        content_field_name (str, optional): The name of the field containing document content.
                                          Defaults to 'content'.

    Returns:
        list: A list of chunk dictionaries. Each chunk contains:
            - All original document fields except the content field
            - 'start': Starting position of the chunk in original content
            - 'content': The chunk content

    Example:
        >>> documents = [{'content': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, size=100, step=50)
        >>> # Or with custom content field:
        >>> documents = [{'text': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, content_field_name='text')
    """
    results = []

    for doc in documents:
        doc_copy = doc.copy()
        doc_content = doc_copy.pop(content_field_name)
        chunks = sliding_window(doc_content, size=size, step=step)
        for chunk in chunks:
            chunk.update(doc_copy)
        results.extend(chunks)

    return results

In [216]:
chunks = chunk_documents(parsed_data, 
                         size=30,
                         step=15,
                         content_field_name='text')

In [217]:
# Number of chunks
len(chunks)

9088

**Q: Question 5. Number of chunks**   
A: There are 9088 chunks

### Step 4: Indexing with MinSearch

In [190]:
from minsearch import Index

# Doing for minsearch
for chunk in chunks:
    chunk['content'] = str(chunk['content'])
    chunk['_metadata'] = str(chunk['metadata'])


index = Index(
    text_fields=['content', '_metadata'],
)

index.fit(chunks)

In [191]:
search_results = index.search('how do I make money with AI?', num_results=5)

In [199]:
search_results[0]['content']

'["This week, we\'ll talk about volunteering and open source work. We have a special guest today, Sara. Sara is a Google Developer expert in machine learning, a Google PhD fellow, and also a co-founder of AI Wonder Girls. She likes to demystify AI to empower individuals with tools and mindsets that require building solutions that matter to the community and humanity.", \'We met with Sara in October, I think, at a conference in Porto. It was an amazing conference. We had a very nice chat. Sara was talking about what she does and I thought “She would be an amazing guest.” And here we are 3, 4, 5 months after that, finally. [chuckles] So, welcome to the interview.\', \'Thank you.\', None, "The questions for today\'s interview were prepared by Johanna Bayer. As always, thanks, Johanna, for your help. Before we start – before we go into our main topic of open source work and volunteering – let\'s start with your background. Can you tell us about your career journey so far?", "Yeah, sure. I 

In [202]:
search_results[0]["metadata"]

{'filename': '_podcast/s17e07-make-impact-through-volunteering-open-source-work.md',
 'episode_title': 'Make an Impact Through Volunteering Open Source Work',
 'episode_number': 7,
 'ids': {'anchor': 'atatalksclub/episodes/Make-an-Impact-Through-Volunteering-Open-Source-Work---Sara-EL-ATEIF-e2g4dan',
  'youtube': 'aHdaIwOEI8Q'},
 'image': 'images/podcast/s17e07-make-impact-through-volunteering-open-source-work.jpg',
 'links': {'anchor': 'https://podcasters.spotify.com/pod/show/datatalksclub/episodes/Make-an-Impact-Through-Volunteering-Open-Source-Work---Sara-EL-ATEIF-e2g4dan',
  'apple': 'https://podcasts.apple.com/us/podcast/make-an-impact-through-volunteering-open-source-work/id1541710331?i=1000646627892',
  'spotify': 'https://open.spotify.com/episode/7tZSSgv1yAlnoMyB4ggQmb?si=AqDaME2QS26usoZjOEWNtQ',
  'youtube': 'https://www.youtube.com/watch?v=aHdaIwOEI8Q'},
 'season': 17,
 'short': 'Make an Impact Through Volunteering Open Source Work',
 'guests': ['saraelateif']}

In [204]:
chunks[0]

{'start': 0,
 'content': '["This week we\'ll talk about being a data generalist. We\'ll discuss going from bioinformatics to freelancing. We have a special guest today, Katya. As a freelancer Katya is helping companies bridge the gap between business and data by building actionable analytics and coaching the teams. She has a lot of broad experience in startups, entrepreneurship and scale-ups. Katya was head of analytics at Gitti, a beauty brand. She tried to start her own fintech business with Entrepreneur First and she worked as a data scientist at Zalando. Welcome to the show. It\'s a pleasure to have you here.", "Yes, thank you so much for the invitation. It was really nice to catch up, actually. I think we\'ve known each other for some time. I\'m really happy to be here.", None, "I tried to invite you multiple times. Finally, we managed to do this. [chuckles] Before we start with our main topic of being a data generalist, let\'s start with your background. Can you tell us about you